# Chapter 14 — Text Classification (NLP): **from scratch AND with Keras**

Maps to Chollet Ch.14. Text isn't numeric, so every NLP model starts with the same pipeline:

**standardize → split into tokens → index to integers → feed a model**

Two ways to represent a document once tokenized:
- **Set model (bag-of-words)** — throw away word order, just "which words appear." Simple, fast, strong baseline.
- **Sequence model** — keep order, process word-by-word (Embedding + LSTM, or later, Transformer).

Part A builds the pipeline **from scratch** (your teacher's emphasis); Part B does it the production way with Keras.

## Get real review text (offline & fast)
Chollet uses raw IMDb text files. To keep this notebook runnable offline, we **decode** Keras's
integer-encoded IMDb back into real review strings. (The raw-directory workflow is in the templates.)

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, re, collections
from keras import layers

(xi_tr, y_tr), (xi_te, y_te) = keras.datasets.imdb.load_data(num_words=20000)
wi = keras.datasets.imdb.get_word_index()
inv = {v+3: k for k, v in wi.items()}; inv[0]=""; inv[1]=""; inv[2]="?"   # 0,1,2 reserved
def decode(seq): return " ".join(inv.get(i, "?") for i in seq).strip()

N, M = 4000, 1000                                    # subset -> fast; raise on Colab
train_texts = [decode(s) for s in xi_tr[:N]]; train_lab = y_tr[:N].astype("float32")
test_texts  = [decode(s) for s in xi_te[:M]]; test_lab  = y_te[:M].astype("float32")
print("a review:", train_texts[0][:90], "...")
print("label (1=pos,0=neg):", train_lab[0])


---
# Part A — From scratch (NLP mechanics by hand)

## A1. The tokenizer: standardize → split → index
A tokenizer is just three steps wrapped in a class. We do word-level here (regex splits words + punctuation).

In [ ]:
def split_words(text):
    return re.findall(r"[\w]+|[.,!?;]", text.lower())   # standardize(lower)+split in one

class WordTokenizer:
    def __init__(self, vocabulary):
        self.vocab = vocabulary; self.unk = vocabulary["[UNK]"]
    def __call__(self, text):
        return [self.vocab.get(tok, self.unk) for tok in split_words(text)]   # index

print(split_words("The quick brown fox jumped over the lazy dog."))


## A2. Build a vocabulary (most-common tokens + `[UNK]`)
Limit vocab size to the most frequent tokens; everything else maps to `[UNK]` (index 0). Smaller vocab =
fewer model parameters.

In [ ]:
def build_vocab(texts, max_size=2000):
    cnt = collections.Counter()
    for t in texts: cnt.update(split_words(t))
    vocab = {"[UNK]": 0}
    for tok, _ in cnt.most_common(max_size - 1):
        vocab[tok] = len(vocab)
    return vocab

vocab = build_vocab(train_texts, max_size=2000)
tok = WordTokenizer(vocab)
print("vocab size:", len(vocab))
print("most common after UNK:", list(vocab)[1:8])
print("indexed:", tok("a brilliant movie with zzzqq")[:8], " (zzzqq -> 0 = UNK)")


## A3. Subword tokenization — Byte-Pair Encoding (what GPT uses)
Word vocab explodes on big corpora; char vocab makes huge sequences. **BPE** bridges both: start from
characters, then repeatedly **merge the most frequent adjacent pair** into one symbol. Common words merge
fully; rare words stay split. Here's the merge loop on a toy corpus.

In [ ]:
def count_split(data):
    counts = collections.Counter()
    for line in data:
        for word in re.findall(r"[\w]+", line.lower()):
            counts[" ".join(word)] += 1          # "brown" -> "b r o w n"
    return dict(counts)

def count_pairs(counts):
    pairs = collections.Counter()
    for wsp, f in counts.items():
        s = wsp.split()
        for pr in zip(s[:-1], s[1:]): pairs[pr] += f
    return pairs

data = ["the quick brown fox", "the slow brown fox", "the quick brown foxhound"]
counts = count_split(data)
for step in range(6):
    pairs = count_pairs(counts)
    if not pairs: break
    first, second = max(pairs, key=pairs.get)               # most frequent pair
    counts = {re.sub(rf"(?<!\S){first} {second}(?!\S)", first+second, k): v
              for k, v in counts.items()}
    print(f"after merge {step+1} ({first}+{second}):", list(counts.keys()))


## A4. Bag-of-words vectorization, by hand
A **set model** turns each document into a fixed-length 0/1 vector: position *v* = 1 if vocab word *v* is
present. Word order is discarded. This is exactly what a multi-hot encoding is.

In [ ]:
def bag_of_words(texts, vocab):
    X = np.zeros((len(texts), len(vocab)), dtype="float32")
    for i, t in enumerate(texts):
        for tok in split_words(t):
            X[i, vocab.get(tok, 0)] = 1.0
    return X

Xtr = bag_of_words(train_texts, vocab)
Xte = bag_of_words(test_texts,  vocab)
print("BoW matrix:", Xtr.shape, "(docs x vocab)")


## A5. A from-scratch sentiment classifier (logistic regression + gradient descent)
One neuron: `p = σ(x·w + b)`. Binary cross-entropy gradient is beautifully simple: `∂L/∂(w) = Xᵀ(p − y)/n`.
This is a perceptron with a sigmoid + the BCE loss — pure NumPy, ties straight to Ch.4.

In [ ]:
def sigmoid(z): return 1.0/(1.0 + np.exp(-z))

w = np.zeros(Xtr.shape[1]); b = 0.0; lr = 0.5
for epoch in range(300):
    p = sigmoid(Xtr @ w + b)
    grad = p - train_lab                       # BCE+sigmoid -> (p - y)
    w -= lr * (Xtr.T @ grad) / len(p)
    b -= lr * grad.mean()
    if epoch % 100 == 0:
        loss = -np.mean(train_lab*np.log(p+1e-9) + (1-train_lab)*np.log(1-p+1e-9))
        print(f"epoch {epoch:3d}  BCE={loss:.4f}")

acc = ((sigmoid(Xte @ w + b) > 0.5) == test_lab).mean()
print(f"\nFROM-SCRATCH logistic-regression BoW test accuracy = {acc:.3f}  (baseline 0.5)")

# bonus: the words the model thinks are most positive / negative
top = np.argsort(w); id2tok = {i:t for t,i in vocab.items()}
print("most NEGATIVE words:", [id2tok[i] for i in top[:6]])
print("most POSITIVE words:", [id2tok[i] for i in top[-6:]])


---
# Part B — With Keras
`TextVectorization` does standardize→split→index→vectorize for you, as a layer you `adapt()` to your data.

## B1. Set model — `TextVectorization(multi_hot, ngrams=2)` + Dense (bag-of-bigrams)
`multi_hot` = the BoW vector from A4, automatically. `ngrams=2` also counts adjacent **word pairs**
("not good" becomes a feature), recovering a little word-order info. This simple model is a *very* strong
baseline — often beats fancier ones on smaller data.

In [ ]:
vec_set = layers.TextVectorization(max_tokens=20000, output_mode="multi_hot", ngrams=2)
vec_set.adapt(train_texts)                        # learns the vocabulary from training text
Xtr_s = vec_set(train_texts); Xte_s = vec_set(test_texts)

set_model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),        # binary sentiment
])
set_model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
set_model.fit(Xtr_s, train_lab, epochs=3, batch_size=32, validation_split=0.2, verbose=0)
print("SET (bag-of-bigrams) test acc:",
      set_model.evaluate(Xte_s, test_lab, verbose=0, return_dict=True)["accuracy"])


## B2. Sequence model — `TextVectorization(int)` + Embedding + Bidirectional LSTM
Keep word order. `output_mode="int"` gives integer sequences; an **Embedding** layer maps each token id to
a learned dense vector; a **Bidirectional LSTM** reads the sequence both directions. `mask_zero=True` tells
the model to ignore padding.

In [ ]:
vec_seq = layers.TextVectorization(max_tokens=20000, output_mode="int",
                                   output_sequence_length=200)
vec_seq.adapt(train_texts)
Xtr_q = vec_seq(train_texts); Xte_q = vec_seq(test_texts)

seq_model = keras.Sequential([
    layers.Embedding(20000, 32, mask_zero=True),  # token id -> 32-dim vector
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(1, activation="sigmoid"),
])
seq_model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
seq_model.fit(Xtr_q, train_lab, epochs=2, batch_size=32, validation_split=0.2, verbose=0)
print("SEQUENCE (BiLSTM) test acc:",
      seq_model.evaluate(Xte_q, test_lab, verbose=0, return_dict=True)["accuracy"])


## B3. Sets vs sequences — which to use?
| | Set (bag-of-words/bigrams) | Sequence (Embedding+LSTM/Transformer) |
|---|---|---|
| word order | discarded | preserved |
| data needed | works on small data | needs more data to shine |
| speed | very fast | slower |
| when | strong baseline, short texts, small data | large data, long-range meaning, SOTA |

**Rule of thumb (Chollet):** start with bag-of-bigrams. Only move to sequence models if you have lots of
data and the set model plateaus. On our tiny 4k subset the set model wins — the LSTM needs more data/epochs.

> Chapter 15 replaces the LSTM with the **Transformer**, which is order-aware *and* parallelizable — the
> architecture behind modern LLMs.

---
# ✍️ PROBLEMS

### P1 — Char & subword tokenizers from scratch
Write a `CharTokenizer` (split on every character) and compare sequence length vs the `WordTokenizer` on one
review. Then extend the BPE demo (A3) into a real `compute_bpe_vocab(texts, vocab_size)` that returns a
vocab + merge list, and tokenize a sentence with it.

In [ ]:
# TODO


### P2 — Improve the from-scratch classifier
Add **L2 regularization** to the from-scratch logistic regression (gradient gets `+ lambda*w`). Sweep
`lambda ∈ {0, 1e-3, 1e-2}` and report test accuracy. Then switch BoW from binary presence to **counts**
(how many times each word appears) — does it help?

In [ ]:
# TODO


### P3 — TF-IDF set model
Re-run B1 with `output_mode="tf_idf"` instead of `multi_hot`. Does TF-IDF beat plain multi-hot bigrams on
the test set? Also try `ngrams=1` vs `ngrams=2` vs `ngrams=3`.

In [ ]:
# TODO


### P4 — Make the sequence model win
The BiLSTM underperforms on 4k samples. Increase to N=15000 train reviews, train 5 epochs with
EarlyStopping, and add a second LSTM/Dropout. Can you get the sequence model to beat the set model? Report
both with a fair comparison.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — From-scratch text pipeline (tokenizer + vocab + BoW)

In [ ]:
import re, collections, numpy as np
def split_words(t): return re.findall(r"[\w]+|[.,!?;]", t.lower())
def build_vocab(texts, max_size=20000):
    cnt=collections.Counter()
    for t in texts: cnt.update(split_words(t))
    v={"[UNK]":0}
    for tok,_ in cnt.most_common(max_size-1): v[tok]=len(v)
    return v
def bag_of_words(texts, vocab):
    X=np.zeros((len(texts),len(vocab)),"float32")
    for i,t in enumerate(texts):
        for tok in split_words(t): X[i,vocab.get(tok,0)]=1.0
    return X


### T2 — From-scratch logistic-regression classifier (BCE + gradient descent)

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))
w=np.zeros(X.shape[1]); b=0.0; lr=0.5
for _ in range(300):
    p=sigmoid(X@w+b); g=p-y
    w-=lr*(X.T@g)/len(p); b-=lr*g.mean()
pred=(sigmoid(Xtest@w+b)>0.5).astype(int)


### T3 — Keras set model (bag-of-bigrams) — strong baseline, use first

In [ ]:
from keras import layers
import keras
vec = layers.TextVectorization(max_tokens=20000, output_mode="multi_hot", ngrams=2)
vec.adapt(train_texts)                       # list/Dataset of raw strings
model = keras.Sequential([layers.Dense(16, activation="relu"),
                          layers.Dense(1,  activation="sigmoid")])
model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
model.fit(vec(train_texts), train_labels, epochs=10, validation_split=0.2)
# multiclass text -> Dense(C, "softmax") + "sparse_categorical_crossentropy"


### T4 — Keras sequence model (Embedding + BiLSTM)

In [ ]:
vec = layers.TextVectorization(max_tokens=20000, output_mode="int", output_sequence_length=250)
vec.adapt(train_texts)
model = keras.Sequential([
    layers.Embedding(20000, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])
model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
model.fit(vec(train_texts), train_labels, epochs=10, validation_split=0.2)


### T5 — Loading RAW text files from folders (the real-world / Colab path)

In [ ]:
# Folder layout:  data/train/pos/*.txt , data/train/neg/*.txt , data/val/... , data/test/...
from keras.utils import text_dataset_from_directory
train_ds = text_dataset_from_directory("data/train", batch_size=32)
val_ds   = text_dataset_from_directory("data/val",   batch_size=32)
test_ds  = text_dataset_from_directory("data/test",  batch_size=32)
# adapt vectorizer on TEXT ONLY, then map the dataset:
vec.adapt(train_ds.map(lambda x, y: x))
train_v = train_ds.map(lambda x, y: (vec(x), y))
# model.fit(train_v, validation_data=val_ds.map(lambda x,y:(vec(x),y)), epochs=10)


---
### ✅ Checklist
- [ ] Explain the pipeline standardize → split → index, and word vs char vs subword (BPE) trade-offs.
- [ ] Implement a word tokenizer, a vocabulary, and bag-of-words from scratch.
- [ ] Run BPE merges by hand and explain why subword tokenization dominates modern LLMs.
- [ ] Train a from-scratch logistic-regression text classifier and beat the 0.5 baseline.
- [ ] Build both a Keras set model (multi_hot bigrams) and sequence model (Embedding+BiLSTM).
- [ ] Decide sets vs sequences based on data size; know bag-of-bigrams is the default baseline.

This is the last chapter in your **Ch.7–14 syllabus**. If you want, I can also build the from-scratch
**CNN** (your lab's CNN-from-scratch) and **Kohonen SOM** notebooks, or a **mock contest** that mixes
tabular / image / text tasks under time pressure. Tell me which.